In [2]:
import pandas as pd
import os
import sys

print("Python version:", sys.version)
print("Pandas version:", pd.__version__)

Python version: 3.11.13 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 13:03:15) [MSC v.1929 64 bit (AMD64)]
Pandas version: 2.3.0


In [2]:
# fORMAT IS 
# ID	Target	PAM Index
# Guide_1	ATGCTAGCTAGGGCATGAGGCATGCTAGTGACTGCATGGTAC	17
# Guide_2	ATCGATGACTGATCGTAGCTAGCTGGGATGCTAGCTAGTTGCATGCTAGGAGTCAGCTAG	23
# Guide_3	GATAGTCGTAGGCTAGCTAGCTAGCTGGCAAGTGTGGAAAAGGGGATGCATGTA	25

In [3]:
DATA_PATH = "../data" # could be "../data/experimental"
MODEL_FORMATTING_PATH = "../data/model_formatting/FORECasT/"


In [4]:
# list all csv files in the DATA_PATH
original_datasets = [f for f in os.listdir(DATA_PATH) if f.endswith('.csv')]
print(len(original_datasets))

# Remove LINDEL because the tool doesn't predict on 'N'ArithmeticError
original_datasets = [f for f in original_datasets if 'Lindel' not in f]
print(len(original_datasets))

# add to them the folder path
original_datasets = [os.path.join(DATA_PATH, f) for f in original_datasets]
print(original_datasets[0])

22
21
../data\ALDIT_HAP1.csv


In [5]:
# get all the targets from all the datasets
all_targets_and_pam_positions = set()
for dataset in original_datasets:
    df = pd.read_csv(dataset)
    print(f"Loaded {dataset} with {len(df)} entries.")
    targets = df['sequence'].tolist()
    pam_positions = df['PAM position'].tolist()
    for target, pam_position in zip(targets, pam_positions):
        all_targets_and_pam_positions.add((target, pam_position))

print(f"Total unique targets collected: {len(all_targets_and_pam_positions)}")

Loaded ../data\ALDIT_HAP1.csv with 15212 entries.
Loaded ../data\ALDIT_Jurkat.csv with 65050 entries.
Loaded ../data\ALDIT_Jurkat_DNTTKO.csv with 10188 entries.
Loaded ../data\ALDIT_K562.csv with 79964 entries.
Loaded ../data\ALDIT_K562_DNTTOE.csv with 14076 entries.
Loaded ../data\FORECasT_BOB.csv with 35065 entries.
Loaded ../data\FORECasT_CHO.csv with 35129 entries.
Loaded ../data\FORECasT_HAP1.csv with 35128 entries.
Loaded ../data\FORECasT_K562.csv with 35129 entries.
Loaded ../data\FORECasT_K562_2A_TREX2.csv with 35129 entries.
Loaded ../data\FORECasT_K562_eCAS9.csv with 34518 entries.
Loaded ../data\FORECasT_K562_TREX2.csv with 35129 entries.
Loaded ../data\FORECasT_mESC.csv with 35104 entries.
Loaded ../data\FORECasT_RPE1.csv with 34671 entries.
Loaded ../data\SPROUT_T.csv with 1603 entries.
Loaded ../data\XCRISP_FORECasT_HAP1.csv with 10827 entries.
Loaded ../data\XCRISP_FORECasT_mESC.csv with 10794 entries.
Loaded ../data\XCRISP_FORECasT_TREX2.csv with 10500 entries.
Loaded .

In [6]:
MERGED_FORMAT_PATH = os.path.join(MODEL_FORMATTING_PATH, "merged.txt")

with open(MERGED_FORMAT_PATH, 'w') as f:
    # write the header
    f.write("ID\tTarget\tPAM Index\n")
    for i, (sequence, pam_position) in enumerate(all_targets_and_pam_positions):
        guide_id = f"Guide_{i + 1}"
        target_sequence = sequence
        pam_position = pam_position  # Use 'N/A' if PAM_POSITION is not present
        f.write(f"{guide_id}\t{target_sequence}\t{pam_position}\n")

Now clone  
https://github.com/felicityallen/SelfTarget


navigate to /indel_prediction_predictor/

Put the CROP/data/model_formatting/FORECast/merged.txt into   

SelfTarget/indel_prediction/predictor/


Now run the following command:

```bash
  docker run -d --name forecast_job \
  --entrypoint /bin/bash \
  --user $(id -u):$(id -g) \
  -v "${PWD}:/app" \
  -w /app \
  quay.io/felicityallen/selftarget \
  -c "python FORECasT.py merged.txt merged_output"
```

Check the logs using
```bash
docker logs -f forecast_job
```

Check appoximate progress (Assuming 1 line per sequence) using
```bash
docker logs forecast_job 2>&1 | wc -l
```

For us we got ~X predictions per second

And finally

```bash
docker rm forecast_job
```

We now need to reconstruct the predictions for each of the datasets

In [3]:
# create a folder named 'FORECasT_preds' in current directory
import os
OUTPUT_PREDICTIONS_PATH = "FORECasT_preds"
if not os.path.exists(OUTPUT_PREDICTIONS_PATH):
    os.makedirs(OUTPUT_PREDICTIONS_PATH)

In this folder put `merged_output_predictedindelsummary.txt` (`merged_output_predictedreads.txt` can be deleted) which is obtained from FORECasT

In [2]:
import pandas as pd
import os
import re
import glob

# Paths - Update these to your actual directory structure
PREDS_DIR = "./FORECasT_preds/"
MERGED_TXT = os.path.join(PREDS_DIR, "merged.txt")
SUMMARY_TXT = os.path.join(PREDS_DIR, "merged_output_predictedindelsummary.txt")
DATA_DIR = "../data/"
OUTPUT_DIR = "./FORECasT_preds/"

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# --- Step 1: Map Guide_ID to (Sequence, PAM_Position) ---
# merged.txt format: ID \t Target \t PAM Index
guide_to_seq = {}
mapping_df = pd.read_csv(MERGED_TXT, sep='\t')
for _, row in mapping_df.iterrows():
    guide_to_seq[row['ID']] = (row['Target'], int(row['PAM Index']))

# --- Step 2: Parse Predicted Indel Summary ---
# We calculate the FS ratio per guide
forecast_results = {} # {(seq, pam): fs_ratio}

with open(SUMMARY_TXT, 'r') as f:
    current_guide = None
    total_fs_count = 0
    total_indel_count = 0
    
    for line in f:
        line = line.strip()
        if not line: continue
        
        # New Guide Block starts with @@@
        if line.startswith("@@@"):
            # Save previous guide results before starting new one
            if current_guide and total_indel_count > 0:
                forecast_results[guide_to_seq[current_guide]] = total_fs_count / total_indel_count
            
            # Reset for new guide
            current_guide = line.split('\t')[0].replace('@@@', '')
            total_fs_count = 0
            total_indel_count = 0
            continue
            
        parts = line.split('\t')
        indel_id = parts[0]
        count = float(parts[2])
        
        # Skip wildtype/no-change
        if indel_id == "-":
            continue
            
        # Calculate Delta Length
        # Logic: D is negative, I is positive. Sum them up.
        # Example: D29_L-21C11R20 -> -29
        matches = re.findall(r'([DI])(\d+)', indel_id)
        delta_length = 0
        for m_type, m_len in matches:
            val = int(m_len)
            if m_type == 'D':
                delta_length -= val
            else:
                delta_length += val
        
        # Check for Frameshift
        if delta_length % 3 != 0:
            total_fs_count += count
        total_indel_count += count

    # Save the very last guide in the file
    if current_guide and total_indel_count > 0:
        forecast_results[guide_to_seq[current_guide]] = total_fs_count / total_indel_count

# --- Step 3: Map results back to original CSVs ---
csv_files = glob.glob(os.path.join(DATA_DIR, "*.csv"))
# Filter out Lindel if necessary as you did before
csv_files = [f for f in csv_files if 'Lindel' not in f]

print(f"Mapping FORECasT results to {len(csv_files)} files...")

for file_path in csv_files:
    file_name = os.path.basename(file_path)
    df = pd.read_csv(file_path)
    
    # Map the ratio using the (sequence, PAM position) tuple
    df['FORECasT_FS_ratio'] = df.apply(
        lambda row: forecast_results.get((row['sequence'], int(row['PAM position']))), 
        axis=1
    )
    
    output_path = os.path.join(OUTPUT_DIR, file_name)
    df.to_csv(output_path, index=False)
    print(f"Saved: {file_name}")

print("\nProcessing complete.")

Mapping FORECasT results to 22 files...
Saved: ALDIT_HAP1.csv
Saved: ALDIT_Jurkat.csv
Saved: ALDIT_Jurkat_DNTTKO.csv
Saved: ALDIT_K562.csv
Saved: ALDIT_K562_DNTTOE.csv
Saved: FORECasT_BOB.csv
Saved: FORECasT_CHO.csv
Saved: FORECasT_HAP1.csv
Saved: FORECasT_K562.csv
Saved: FORECasT_K562_2A_TREX2.csv
Saved: FORECasT_K562_eCAS9.csv
Saved: FORECasT_K562_TREX2.csv
Saved: FORECasT_mESC.csv
Saved: FORECasT_RPE1.csv
Saved: SPROUT_T.csv
Saved: SPROUT_T_CROTON_VERSION.csv
Saved: XCRISP_FORECasT_HAP1.csv
Saved: XCRISP_FORECasT_mESC.csv
Saved: XCRISP_FORECasT_TREX2.csv
Saved: XCRISP_inDelphi_mESC.csv
Saved: XCRISP_inDelphi_mESC_NHEJdeficient.csv
Saved: XCRISP_inDelphi_U2OS.csv

Processing complete.
